# NAMD-FEP — notebook 3 · the hybrid ligand and the two systems

Three commands, each followed by its visualisation:

| step | command | writes | look at |
|---|---|---|---|
| ① | `rbfe hybrid` | `hybrid/hybrid.{pdb,rtf,prm}` | the B-factor marks −1/0/+1 |
| ② | `rbfe build` | `complex/`, `solvent/` PSF/PDB + `box.json` | the solvated boxes |
| ③ | `rbfe inputs` | `ionized.fep`, the `.namd` configs, `fep.tcl` | the marked atoms |

plus `rbfe check`, which parses every config with NAMD's `--dryrun` so a mistake
costs seconds rather than a GPU-hour.

> Steps ② and the `--dryrun` in ③ need **VMD/psfgen and NAMD 3.x**, which are not
> installable on Binder. Those cells detect the binaries and **skip themselves with
> instructions** when they are absent — everything up to that point, and the
> understanding of what they produce, works everywhere.

## 3.1 Setup

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path.cwd().resolve()
if not (REPO / "bin" / "rbfe").exists():
    REPO = REPO.parent
os.environ["PATH"] = str(REPO / "bin") + os.pathsep + os.environ["PATH"]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

SYS = REPO / "_tutorial" / "6i5i"
assert (SYS / "system.ini").exists(), "run notebook 02 first"

def run_rbfe(*args, check=True):
    """One rbfe subcommand inside the system directory, output echoed."""
    print("$", "rbfe", *args)
    r = subprocess.run(["rbfe", *args], cwd=SYS, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        if check:
            raise RuntimeError(f"rbfe {args[0]} failed ({r.returncode})")
    return r

NAMD = os.environ.get("NAMD") or shutil.which("namd3")
VMD  = os.environ.get("VMD")  or shutil.which("vmd")
print("NAMD:", NAMD or "not found -- build/run cells will skip")
print("VMD :", VMD  or "not found -- build/run cells will skip")

## 3.2 ① `rbfe hybrid` — the dual-topology hybrid ligand

NAMD's dual-topology scheme keeps **both** ligands in one system: atoms common to
both appear once, atoms that differ appear twice and are switched on/off with λ.
Which is which is encoded in the hybrid PDB's **B-factor column** — the single most
important convention in this repository:

| B-factor | role | behaviour |
|---:|---|---|
| **−1** | A-only (vanishing) | off as λ → 1 |
| **0** | common core + the whole environment | always on |
| **+1** | B-only (appearing) | on as λ → 1 |

**Read the report the command prints.** Four lines matter: the atom mapping (did it
find the atoms you meant?), charge closure (λ=0 must be the reference's net charge,
λ=1 the mutant's), the parameter counts, and any term with no source parameter.

In [ ]:
def inputs_ready():
    d = SYS / "inputs"
    need = ["protein.pdb"] + [f"{s}.{e}" for s in ("ref", "mut")
                              for e in ("pdb", "rtf", "prm")]
    missing = [f for f in need if not (d / f).exists()]
    if missing:
        print(f"inputs missing: {missing}\n"
              f"  run the parameterisation cell of notebook 02 (needs acpype+obabel)")
        return False
    return True

if inputs_ready():
    run_rbfe("hybrid")
    print("wrote:", sorted(p.name for p in (SYS / "hybrid").iterdir()))

## 3.3 Inspect the hybrid — the 60 seconds that save a GPU day

The B-factor column **is** the contract for everything downstream. Colour the hybrid
by it: blue = common core (0), red = vanishing (−1), green = appearing (+1). Confirm
the marks are where you expect **before** any GPU time is spent.

In [ ]:
import nglview as nv

hybrid_pdb = SYS / "hybrid" / "hybrid.pdb"

if not hybrid_pdb.exists():
    print("no hybrid yet -- the `rbfe hybrid` cell above was skipped\n"
          "(missing parameterised inputs).")
else:
    # parse the marks for a text cross-check
    marks = {"vanish (-1)": [], "common (0)": [], "appear (+1)": []}
    for line in hybrid_pdb.read_text().splitlines():
        if line.startswith(("ATOM  ", "HETATM")):
            b = float(line[60:66])
            key = "vanish (-1)" if b < 0 else "appear (+1)" if b > 0 else "common (0)"
            marks[key].append(line[12:16].strip())
    for k, v in marks.items():
        print(f"{k:<12} {len(v):>3} atoms  {v if len(v) <= 8 else ''}")

    view = nv.show_structure_file(str(hybrid_pdb))
    view.layout.width = "100%"
    view.layout.height = "600px"  # fixed canvas: blank on JupyterLab without it
    view.clear_representations()
    view.add_representation("ball+stick", selection="all", color="bfactor")
    view.center()
    print("\ncoloured by B-factor: blue = common, red = vanishing, ... (ngl bfactor scale)")
    display(view)
    view.handle_resize()  # force the stage to recalc its size once visible


## 3.4 ② `rbfe build` — psfgen → solvate → ionize

VMD/psfgen builds the two legs — **complex** (protein + hybrid ligand, split into
segments at numbering breaks) and **solvent** (the hybrid alone) — solvates each in
a water box padded 15 Å, and neutralises with ions. Writes `<leg>/ionized.{psf,pdb}`
and `box.json` (the cell vectors step ③ reads back — the cell is never a constant).

Takes minutes. **Skipped on Binder** (no VMD); run it on the machine that has VMD.

In [ ]:
if VMD is None:
    print("VMD not found -- skipping the build.\n"
          "On the GPU box: cd", SYS, "&& rbfe build\n"
          "The shipped 6I5I example (6I5I_DUAL_FEP/) shows the expected outputs.")
else:
    run_rbfe("build")

## 3.5 Look at the solvated systems

The complex is ~65 000 atoms — too heavy for an interactive widget at full detail,
so the view shows the protein surface, the ligand, and the box outline; the solvent
leg is small enough to show every water. When the build was skipped, this cell shows
nothing and says so.

In [ ]:
import json

for leg in ("complex", "solvent"):
    pdb = SYS / leg / "ionized.pdb"
    box = SYS / leg / "box.json"
    if not pdb.exists():
        print(f"{leg}: not built yet")
        continue
    if box.exists():
        print(f"{leg} box:", json.loads(box.read_text()))
    view = nv.show_structure_file(str(pdb))
    view.layout.width = "100%"
    view.layout.height = "600px"  # fixed canvas: blank on JupyterLab without it
    view.clear_representations()
    if leg == "complex":
        view.add_representation("surface", selection="protein",
                                color="lightgrey", opacity=0.6)
        view.add_representation("ball+stick", selection="not protein and not water and not ion",
                                color="bfactor")
    else:
        view.add_representation("ball+stick", selection="not water and not ion",
                                color="bfactor")
        view.add_representation("line", selection="water", opacity=0.3)
    view.add_representation("unitcell")
    view.center()
    display(view)
    view.handle_resize()  # force the stage to recalc its size once visible


## 3.6 ③ `rbfe inputs` — the NAMD runtime files

Writes, per leg: `ionized.fep` (the solvated PDB with the `alchFile` B-factor column
set — ligand −1/0/+1, environment 0), the four `.namd` configs (`nvt_equil`,
`npt_equil`, `md_forward`, `md_backward`), and once: `fep.tcl` with the λ schedule.

Two load-bearing checks happen here: the marked atoms are matched **against the
built system** (psfgen upper-cases names; an exact-match miss would silently write a
one-directional alchemical file), and the cell vectors come from `box.json`.

In [ ]:
if VMD is None or not (SYS / "complex" / "ionized.pdb").exists():
    print("skipping -- `rbfe build` has not run here.")
else:
    run_rbfe("inputs")
    print("\nconfigs:", sorted(p.name for p in (SYS / "complex").glob("*.namd")))

## 3.7 `rbfe check` — parse every config before burning GPU time

NAMD's `--dryrun` reads and validates a config without integrating, so a mistake
costs seconds. With `--smoke` it also runs a short `nvt_equil` and projects the full
job's wall-clock from the `TIMING:` line (never the cumulative `PERFORMANCE:` line).

In [ ]:
if NAMD is None or not (SYS / "fep.tcl").exists():
    print("skipping -- needs NAMD and `rbfe inputs`.\n"
          "On the GPU box: cd", SYS, "&& rbfe check --smoke")
else:
    run_rbfe("check", "--smoke")

## 3.8 Where you are

✔ hybrid ligand built, B-factors inspected · ✔ both systems built (or deferred to
the GPU box) · ✔ runtime configs written and dry-run.

**Next: `04_run_monitor.ipynb`** — launch the production run (GPU) and watch it.